## 智能体

#### 加载环境并初始化

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.config import get_stream_writer
from langchain.agents import create_agent
from rich.pretty import pprint

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """获取指定城市的天气

    Args:
        city: 具体的城市
    """
    writer = get_stream_writer()
    writer({"type": "weather", "city": city})

    return f"{city}天气晴天~"

agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather]
)

messages = [
    SystemMessage("你是查询天气的助手，根据用户的提问查询天气。如果问题跟天气无关，则拒绝回答"),
    HumanMessage("北京天气如何")
]

#### 流式输出

##### 1、updates 模式（默认）

输出每个节点执行后的状态增量，适合观察“模型请求工具 → 工具返回 → 模型回答”的过程

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="updates"):
    pprint(chunk)

##### 2、values 模式

输出每一步后的完整状态，例如当前累积的全部消息，而不只是本次新增内容

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="values"):
    pprint(chunk)

##### 3、messages 模式

输出模型生成的消息片段及元数据，包括文本、工具调用参数等，适合实现打字机效果

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="messages"):
    # pprint(chunk)
    print(chunk[0].content, end="", flush=True)

##### 4、tasks 模式

输出节点任务的开始、结束、结果和错误，用于观察任务执行情况

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="tasks"):
    pprint(chunk)

##### 5、debug 模式

输出更详细的调试信息，综合任务、检查点等执行事件

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="debug"):
    pprint(chunk)

##### 6、checkpoints 模式

输出检查点创建事件，观察保存的执行状态，需要配置 checkpointer

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

custom_agent = create_agent(
    model='deepseek:deepseek-flash',
    tools=[get_weather],
    checkpointer=checkpointer
)

config: RunnableConfig = {"configurable": {"thread_id": "session01"}}
checkpoint_count = 0
for chunk in custom_agent.stream({'messages': messages}, config=config, stream_mode="checkpoints"):
    checkpoint_count += 1
    print(f"检查点 {checkpoint_count}")
    pprint(chunk)

##### 7、custom 模式

输出代码主动发送的自定义数据，例如“已处理 30/100 条”

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode="custom"):
    pprint(chunk)

##### 8、多个模式组合使用

In [ ]:
for chunk in agent.stream({'messages': messages}, stream_mode=["updates", "custom"]):
    pprint(chunk)